## Chatbots With Langgraph

In [1]:
!pip install langgraph langsmith

In [2]:
!pip install langchain langchain_groq langchain_community

In [3]:
# from google.colab import userdata
import os
from dotenv import load_dotenv
load_dotenv()
groq_api_key=os.getenv('sample_qroq_api_key')
langsmith=os.getenv('LANGSMITH_API_KEY')
# print(langsmith)


In [4]:
import os
os.environ["LANGCHAIN_API_KEY"] = langsmith
os.environ["LANGCHAIN_TRACING_V2"]="true"
os.environ["LANGCHAIN_PROJECT"]="CourseLanggraph"

In [5]:
from langchain_groq import ChatGroq

In [6]:
import httpx

llm=ChatGroq(groq_api_key=groq_api_key,model_name="Gemma2-9b-It",http_client=httpx.Client(verify=False))
llm

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x114c34850>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x114cc7890>, model_name='Gemma2-9b-It', model_kwargs={}, groq_api_key=SecretStr('**********'), http_client=<httpx.Client object at 0x114b13e50>)

In [ ]:
llm.invoke("hello")

AIMessage(content='Hello! How can I help you today? 👋\n', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 10, 'total_tokens': 23, 'completion_time': 0.023636364, 'prompt_time': 0.00116927, 'queue_time': 0.246504039, 'total_time': 0.024805634}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--fbca1884-8f40-40f6-b3d4-8206473d7e4d-0', usage_metadata={'input_tokens': 10, 'output_tokens': 13, 'total_tokens': 23})

Failed to get info from https://api.smith.langchain.com: LangSmithConnectionError('Connection error caused failure to GET /info in LangSmith API. Please confirm your internet connection. SSLError(MaxRetryError("HTTPSConnectionPool(host=\'api.smith.langchain.com\', port=443): Max retries exceeded with url: /info (Caused by SSLError(SSLCertVerificationError(1, \'[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:992)\')))"))\nContent-Length: None\nAPI Key: lsv2_********************************************e1')
Failed to batch ingest runs: langsmith.utils.LangSmithConnectionError: Connection error caused failure to POST https://api.smith.langchain.com/runs/batch in LangSmith API. Please confirm your internet connection. SSLError(MaxRetryError("HTTPSConnectionPool(host='api.smith.langchain.com', port=443): Max retries exceeded with url: /runs/batch (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certif

## Start Building Chatbot Using Langgraph

In [8]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph,START,END
from langgraph.graph.message import add_messages

In [9]:
class State(TypedDict):
  # Messages have the type "list". The `add_messages` function
    # in the annotation defines how this state key should be updated
    # (in this case, it appends messages to the list, rather than overwriting them)
  messages:Annotated[list,add_messages]

graph_builder=StateGraph(State)


In [10]:
graph_builder

In [11]:
def chatbot(state:State):
  return {"messages":llm.invoke(state['messages'])}

In [12]:
graph_builder.add_node("chatbot",chatbot)

In [13]:
graph_builder

In [14]:
graph_builder.add_edge(START,"chatbot")
graph_builder.add_edge("chatbot",END)

In [15]:
graph=graph_builder.compile()

In [16]:
from IPython.display import Image, display
try:
  display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
  pass

In [17]:
while True:
  user_input=input("User: ")
  if user_input.lower() in ["quit","q"]:
    print("Good Bye")
    break
  for event in graph.stream({'messages':("user",user_input)}):
    print(event.values())
    for value in event.values():
      print(value['messages'])
      print("Assistant:",value["messages"].content)

dict_values([{'messages': AIMessage(content='Hello! 👋\n\nHow can I help you today? 😊\n', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 10, 'total_tokens': 25, 'completion_time': 0.027272727, 'prompt_time': 0.001309111, 'queue_time': 0.322175719, 'total_time': 0.028581838}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--2fba4968-b082-4f7b-bf89-81b0093864b1-0', usage_metadata={'input_tokens': 10, 'output_tokens': 15, 'total_tokens': 25})}])
content='Hello! 👋\n\nHow can I help you today? 😊\n' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 10, 'total_tokens': 25, 'completion_time': 0.027272727, 'prompt_time': 0.001309111, 'queue_time': 0.322175719, 'total_time': 0.028581838}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'service_tier': 'on_demand', 'finish_reaso